# MARV × Titans — diffing a test-time memory as it reads (Colab T4 / CPU)

MARV's `diff` normally compares two **training** checkpoints, feature by feature.
This notebook points the same primitive at a **Titans neural memory**: a small MLP
whose weights are updated by gradient descent *while the model reads*.

## How Titans works (the part that matters here)

A Titans layer runs two memories side by side:

- **Short-term memory = attention**, over a small sliding window. Precise, but its cost
  and its KV cache grow with the window, so the window stays small.
- **Long-term memory = a small MLP** (`dim → ~4·dim → dim`) whose weights are **not frozen**
  at inference. After every chunk of tokens the model takes a gradient step on that MLP.

The update rule, per token / chunk:

1. **Surprise = gradient.** Project the token to a key `k` and value `v`. The memory's loss is
   `‖M(k) − v‖²`. Its gradient w.r.t. the MLP weights measures how *surprising* the token is
   — a token the memory already predicts well has a small gradient and barely moves the weights.
2. **Momentum.** Instead of the raw gradient, a running `S_t = η_t·S_{t-1} − θ_t·∇loss` — so the
   tokens *after* a surprising one also get written. `η`, `θ` are produced by the outer network,
   per token.
3. **Forget gate = weight decay.** `M_t = (1 − α_t)·M_{t-1} + S_t`. `α_t → 0` keeps the memory,
   `α_t → 1` wipes it. Also data-dependent.

So the memory is doing **online mini-batch gradient descent with momentum and weight decay**,
and every hyperparameter is a learned function of the current token. **Retrieval** is just a
forward pass `M(query)` — no gradient step.

Two more pieces: **persistent memory** (a few input-independent learnable tokens prepended to
the sequence) and the three ways to wire the long-term memory to attention — *Memory as
Context / Gate / Layer*. None of that matters for this notebook; we only touch the long-term
memory MLP.

## What we do with it

`titans-pytorch` exposes the memory's accumulated weight-delta at **every chunk boundary**
(`state.updates`) — a stack of snapshots of the same evolving MLP. That's exactly what a diff
wants two of. So: snapshot early in a document, snapshot at the end, **diff per hidden unit**
(`gate_cos` / `down_cos` / `norm_ratio`, the `marv.diff.FeatureDelta` fields). Then ask what
each changed unit stored, whether chunks collide on the same unit, and how much of an early
write survives (the forget gate, measured per unit).

Runtime: a T4 is plenty; it also runs on CPU (training the memory is seconds either way).

In [ ]:
!pip install -q titans-pytorch matplotlib
!git clone -q -b marv-titan https://github.com/thebnbrkr/marv.git /content/marv
import sys; sys.path.insert(0, '/content/marv/experiments')

import numpy as np, torch, matplotlib.pyplot as plt
from titans_pytorch import NeuralMemory
from titans_memdiff import DIM, HIDDEN, CHUNK, DOC_LEN, train_recall, snapshots, _cos

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
print('device:', device, ' memory MLP:', f'{DIM}->{HIDDEN}->{DIM}', ' document:', DOC_LEN, 'tokens')

## Build two memories: one untrained, one trained on recall

The trained one's *slow / outer* weights are optimised so the memory, queried with token `i`,
returns token `i` (autoassociative recall). Standard backprop — the inner test-time loop is
differentiable. Compare what the diff shows for each.

In [ ]:
mem_raw = NeuralMemory(dim=DIM, chunk_size=CHUNK).to(device)

mem_trained = NeuralMemory(dim=DIM, chunk_size=CHUNK).to(device)
print('training the memory on autoassociative recall...')
train_recall(mem_trained, steps=400, device=device)

## 1. Diff the memory — early in the document vs the end

One `FeatureDelta` per hidden unit. `gate_cos` near 1 = the unit's input direction didn't move;
low or negative = it repointed. `norm_ratio` > 1 = its weights grew over the document, < 1 = decayed.

In [ ]:
def run_diff(mem, seq):
    U0, U1, vals = snapshots(mem, seq)
    g_in, g_out = U0[1], U0[-1]
    d_in, d_out = U1[1], U1[-1]
    gate_cos = _cos(g_in, g_out, axis=0)
    down_cos = _cos(d_in, d_out, axis=1)
    nr = (np.linalg.norm(g_out, axis=0) + 1e-9) / (np.linalg.norm(g_in, axis=0) + 1e-9)
    return dict(U0=U0, U1=U1, vals=vals, gate_cos=gate_cos, down_cos=down_cos, nr=nr)

seq = torch.randn(1, DOC_LEN, DIM, device=device)
R = {'untrained': run_diff(mem_raw, seq), 'trained': run_diff(mem_trained, seq)}

print(f'{"":<11}{"units moved":>12}{"gate_cos med":>14}{"down_cos med":>14}{"norm_ratio":>12}')
for k, r in R.items():
    moved = (r['gate_cos'] < 0.99).sum()
    print(f'{k:<11}{moved:>7} /{HIDDEN:<3}{np.median(r["gate_cos"]):>+14.3f}'
          f'{np.median(r["down_cos"]):>+14.3f}{r["nr"].mean():>12.2f}')

## 2. Write collisions — does each chunk get its own units, or is it all superposed?

For every hidden unit and every chunk, how strongly did that chunk's write land on that unit,
and does the write direction match that chunk's stored *value*. A unit matching **more than
one** chunk's content is a collision — the same neuron holding several unrelated things.

**Caveat (verified by the sanity-check cell below):** the "does the write match the chunk's
content" test aligns a unit's weight-delta against the *chunk-mean value* — the mean of 16
random vectors, which points almost nowhere. So the exact collision **counts** here are rough.
What's solid is the *write-concentration* curve (right plot) and, below, the forgetting result.

In [ ]:
import math
def collisions(r):
    U1, vals = r['U1'], r['vals']
    incr = np.diff(U1, axis=0)
    inorm = np.linalg.norm(incr, axis=2)
    nvc = min(incr.shape[0], math.ceil(DOC_LEN / CHUNK))
    incr, inorm = incr[:nvc], inorm[:nvc]
    cval = vals[:nvc*CHUNK].reshape(nvc, CHUNK, DIM).mean(1)
    cval /= np.linalg.norm(cval, axis=1, keepdims=True) + 1e-9
    idir = incr / (np.linalg.norm(incr, axis=2, keepdims=True) + 1e-9)
    align = np.einsum('khd,kd->kh', idir, cval)
    per_unit = []
    for u in range(HIDDEN):
        strong = [k for k in range(nvc)
                  if inorm[k, u] > 0.5*inorm[:, u].max() and align[k, u] > 0.15]
        per_unit.append(len(strong))
    return np.array(per_unit), inorm

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i, (k, r) in enumerate(R.items()):
    counts, inorm = collisions(r)
    hist = [np.sum(counts == 0), np.sum(counts == 1), np.sum(counts == 2), np.sum(counts >= 3)]
    ax[0].bar(np.arange(4) + i*0.38, hist, width=0.36, label=k)
    # write concentration: sorted per-unit write norm for the first chunk
    s = np.sort(inorm[0])[::-1]; s = s / s.sum()
    ax[1].plot(np.cumsum(s), label=f'{k} (chunk 0 write)')
ax[0].set_xticks(np.arange(4) + 0.19); ax[0].set_xticklabels(['0', '1', '2', '3+'])
ax[0].set_xlabel('chunks whose content this unit carries'); ax[0].set_ylabel('hidden units'); ax[0].legend()
ax[0].set_title('write collisions per unit')
ax[1].plot([0, HIDDEN], [0, 1], 'k--', lw=0.8, label='perfectly uniform')
ax[1].set_xlabel('hidden units (sorted by write size)'); ax[1].set_ylabel('cumulative share of the write')
ax[1].set_title('is one chunk\'s write spread evenly or concentrated?'); ax[1].legend()
plt.tight_layout(); plt.show()

for k, r in R.items():
    counts, _ = collisions(r)
    print(f'{k:<10} units carrying >1 chunk: {np.sum(counts > 1):>3} / {HIDDEN}')

## 3. Forgetting — how much of an early write survives to the end

Take the units the **first chunk** wrote hardest. Track their down-vector across the rest of
the document: does the direction hold, does the magnitude hold? This is the weight-decay
(forget) gate, measured one neuron at a time.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k, r in R.items():
    U1 = r['U1']
    inorm = np.linalg.norm(np.diff(U1, axis=0), axis=2)
    early = np.argsort(inorm[0])[::-1][:20]
    # trajectory of each early unit's down-vector norm across chunks
    traj = np.linalg.norm(U1[:, early, :], axis=2)   # (chunks, 20)
    ax[0].plot(traj.mean(1), 'o-', label=f'{k} (mean of 20)')
    w_e, w_end = U1[1, early, :], U1[-1, early, :]
    scos = _cos(w_e, w_end, axis=1)
    smag = np.linalg.norm(w_end, axis=1) / (np.linalg.norm(w_e, axis=1) + 1e-9)
    ax[1].scatter(smag, scos, label=k, alpha=0.7)
    print(f'{k:<10} early writes: dir cos {scos.mean():+.2f}  mag ratio {smag.mean():.2f}  '
          f'overwritten {int(np.sum((smag<0.6)|(scos<0.5)))}/20')
ax[0].set_xlabel('chunk'); ax[0].set_ylabel('down-vector norm of the first chunk\'s units')
ax[0].set_title('forgetting curve'); ax[0].legend()
ax[1].axhline(0.5, color='k', lw=0.6); ax[1].axvline(0.6, color='k', lw=0.6)
ax[1].set_xlabel('magnitude retained (end / early)'); ax[1].set_ylabel('direction retained (cos)')
ax[1].set_title('per-unit survival of the first chunk\'s write'); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Sanity checks — is the interpretation above actually supported?

Each claim from the plots, turned into a check with the numbers it rests on. Run it and read
the `PASS` / `FAIL` lines and the CLAIM 4 verdict. The point: separate what's **metric-independent**
(the forgetting curve, `norm_ratio`, write diffuseness) from what leans on the shaky
chunk-mean-value proxy (the exact collision counts).

In [ ]:
# ================== SANITY CHECKS on the figure interpretation ==================
def summarize(mem, seq):
    U0, U1, vals = snapshots(mem, seq)
    incr = np.diff(U1, axis=0)
    nvc  = min(incr.shape[0], DOC_LEN // CHUNK)
    inorm = np.linalg.norm(incr[:nvc], axis=2)                # (nvc, HIDDEN)
    early = np.argsort(inorm[0])[::-1][:20]
    traj  = np.linalg.norm(U1[:, early, :], axis=2).mean(1)
    x = np.sort(inorm[0]); n = len(x)
    gini = (2*np.arange(1, n+1) - n - 1).dot(x) / (n * x.sum() + 1e-12)
    return dict(U1=U1, vals=vals, inorm=inorm, nvc=nvc, traj=traj, gini=gini)

def ok(b): return "PASS" if b else "**FAIL**"

seq = torch.randn(1, DOC_LEN, DIM, device=device)
S = {'untrained': summarize(mem_raw, seq), 'trained': summarize(mem_trained, seq)}

t = S['untrained']['traj']
print("CLAIM 1  untrained: first-chunk units grow / hold, don't forget")
print(f"   norm over chunks: {' '.join(f'{v:.2f}' for v in t)}")
print(f"   end/first ratio = {t[-1]/t[1]:.2f}   -> {ok(t[-1] >= 0.9*t[1])}\n")

t = S['trained']['traj']; r = t[1:]/(t[:-1]+1e-9)
print("CLAIM 2  trained: first-chunk units decay ~exponentially")
print(f"   norm over chunks: {' '.join(f'{v:.2f}' for v in t)}")
print(f"   step ratios:      {' '.join(f'{v:.2f}' for v in r)}")
print(f"   end/peak = {t[-1]/t.max():.3f}   mean step ratio = {r.mean():.2f}")
print(f"   -> {ok(t[-1] < 0.25*t.max() and r.mean() < 1.0)}\n")

print("CLAIM 3  writes spread across all units, not concentrated on a few")
for k in S:
    print(f"   {k:<10} Gini(chunk-0 write) = {S[k]['gini']:.3f}   (0=uniform, >0.6=sparse)")
print(f"   -> {ok(S['untrained']['gini'] < 0.4 and S['trained']['gini'] < 0.4)}\n")

print("CLAIM 4  is the trained collision-drop localization or forgetting?")
for k in S:
    c, _ = collisions(R[k])
    print(f"   {k:<10} carry 0: {int((c==0).sum()):>3}   carry 1: {int((c==1).sum()):>3}   "
          f"carry >1: {int((c>1).sum()):>3}   end/early norm_ratio: {R[k]['nr'].mean():.2f}")
print("   localization -> trained 'carry 1' HIGH, norm_ratio ~1")
print("   forgetting   -> trained 'carry 0' HIGH, norm_ratio << 1")
tc, _ = collisions(R['trained'])
verdict = "FORGETTING" if ((tc==0).sum() > (tc==1).sum() and R['trained']['nr'].mean() < 0.4) else "LOCALIZATION"
print(f"   -> VERDICT: {verdict}\n")

def peak_align(s):
    U1, vals, inorm, nvc = s['U1'], s['vals'], s['inorm'], s['nvc']
    cval = vals[:nvc*CHUNK].reshape(nvc, CHUNK, DIM).mean(1)
    cval /= np.linalg.norm(cval, axis=1, keepdims=True) + 1e-9
    incr = np.diff(U1, axis=0)[:nvc]
    idir = incr / (np.linalg.norm(incr, axis=2, keepdims=True) + 1e-9)
    align = np.einsum('khd,kd->kh', idir, cval)
    pk = inorm.argmax(0)
    strong = inorm.max(0) > np.percentile(inorm.max(0), 50)
    return align[pk, np.arange(HIDDEN)][strong].mean()
print("CLAIM 5  content-alignment of a unit's write AT its peak chunk")
for k in S:
    print(f"   {k:<10} peak-chunk align: {peak_align(S[k]):+.3f}")
print("   NOTE: the target here (chunk-mean of 16 random value vectors) is near-degenerate,")
print("   so ~0 for the trained memory is expected -- this test mostly shows the proxy is weak.\n")

print("CLAIM 6  repeat with 3 fresh document seeds")
for sd in (1, 2, 3):
    torch.manual_seed(sd)
    sq = torch.randn(1, DOC_LEN, DIM, device=device)
    ru, rt = run_diff(mem_raw, sq), run_diff(mem_trained, sq)
    cu, _ = collisions(ru); ct, _ = collisions(rt)
    print(f"   seed {sd}: untrained nr={ru['nr'].mean():.2f} collide>1={int((cu>1).sum()):>3}  |  "
          f"trained nr={rt['nr'].mean():.2f} carry0={int((ct==0).sum()):>3} collide>1={int((ct>1).sum()):>3}")

## What this shows / what to try next

**Solid — metric-independent, stable across seeds (see the sanity-check cell):**

- **Untrained memory does not forget.** The forget gate is effectively off; writes accumulate
  (`norm_ratio` end/early ≈ 1.6–2.0).
- **Trained memory forgets exponentially.** First-chunk write down to ~4% of peak after six
  chunks, half-life ≈ 1 chunk. Tight curve. (Decay *rate* is task-dependent — `train_recall`
  is a crude short-sequence task and may teach a strong gate.)
- **The write is diffuse in both** — Gini 0.04–0.12, near-uniform across all 256 units. No
  sparse "this chunk → these few units" allocation.
- So the trained memory's apparent lack of collisions is **forgetting, not clean storage** —
  by end-of-document it holds almost nothing, so nothing is left to collide.

**Not established:** whether a trained memory *localises* storage at the moment of writing. The
"which chunk did unit u store" test relies on a near-degenerate target (chunk-mean of random
vectors), so its counts are rough and CLAIM 5 comes out ≈ 0.

**Next:**
1. **Answer storage with ablation, not a proxy.** Store *tracked* key→value pairs; ablate
   hidden units one at a time and measure which units' removal kills a given pair's recall
   (`marv.suppress` on the live memory). Overlap of pairs' unit-sets = the real collision.
   Blocker: `titans-pytorch` retrieves per-chunk with per-chunk causal weights — a naive
   `functional_call` on one final weight state only matches the true retrieval at cos ≈ 0.6.
2. **Scale:** `DIM=512`, deeper `NeuralMemory`, longer `DOC_LEN` — does the exponential
   forgetting hold? A survival-vs-distance curve and a capacity knee?
3. **Real vocabulary:** wire the memory into a small LM (titans-pytorch MAC on char enwik8
   fits a T4) so a logit-lens readout works — and fixes the degenerate content target.
4. If it stabilises — a `marv` adapter for a plain-MLP memory, and a real writeup.

See `experiments/README.md` on the `marv-titan` branch.